In [6]:
using NCDatasets, Plots, Colors, FileIO, Dates, Shapefile


In [7]:
# --- 1. Preparation & Metadata Helpers ---
data_folder = "./lake_erie_viirs_data"
files = filter(f -> endswith(f, ".nc"), readdir(data_folder, join=true))
sort!(files)

function get_pretty_date(ds)
    raw_time = ds.attrib["time_coverage_start"]
    dt = DateTime(raw_time[1:10], dateformat"yyyy-mm-dd")
    return Dates.format(dt, "u dd, yyyy")
end

# --- 2. Initialize & Coordinate Sorting ---
ds_init = Dataset(files[1])

# Coalesce missing lon/lat values to NaN so comparisons are safe
raw_lons = coalesce.(ds_init["lon"][:], NaN)
raw_lats = coalesce.(ds_init["lat"][:], NaN)

# Select only finite coordinates inside bounding box
lon_indices = findall(x -> isfinite(x) && -84.0 <= x <= -77.0, raw_lons)
lat_indices = findall(y -> isfinite(y) && 40.0 <= y <= 45.0, raw_lats)

lons = sort(raw_lons[lon_indices])
lats = sort(raw_lats[lat_indices])

# Permutations relative to the selected subset (used to reorder data)
p_lon = sortperm(raw_lons[lon_indices])
p_lat = sortperm(raw_lats[lat_indices])

close(ds_init)

# --- 3. Animation with Plots ---
xtick_vals = collect(-83.0:1.0:-79.0)
ytick_vals = collect(41.5:0.5:43.0)
xtick_labels = ["$(abs(round(x, digits=1)))°W" for x in xtick_vals]
ytick_labels = ["$(round(y, digits=1))°N" for y in ytick_vals]

anim = @animate for file in files
    ds = Dataset(file)
    # Read data and coalesce missing values to NaN before processing
    data = coalesce.(ds["chlor_a"][lon_indices, lat_indices], NaN)
    data_sorted = data[p_lon, p_lat]
    data_sorted[.!isfinite.(data_sorted)] .= 0.1
    z = log10.(data_sorted)

    heatmap(
        lons, lats, z';
        xlims = (-83.6, -78.2),
        ylims = (41.3, 43.1),
        xlabel = "Longitude",
        ylabel = "Latitude",
        title = "Lake Erie Chlorophyll: $(get_pretty_date(ds))",
        aspect_ratio = :equal,
        color = :algae,
        clims = (log10(0.1), log10(50.0)),
        colorbar_title = "Chlorophyll-a (mg/m³)",
        xticks = (xtick_vals, xtick_labels),
        yticks = (ytick_vals, ytick_labels),
        legend = false
    )

    close(ds)
end

mp4(anim, "lake_erie_final_dates.mp4", fps = 2)


[ Info: Saved animation to c:\Users\kavin\OneDrive\Research\CPS\cdc26_ive_intro\code\habs\lake_erie_final_dates.mp4


Plots.AnimatedGif("c:\\Users\\kavin\\OneDrive\\Research\\CPS\\cdc26_ive_intro\\code\\habs\\lake_erie_final_dates.mp4")

In [18]:
# Load the shapefile
shapefile_path = "./ne_10m_lakes.shp"
lakes = Shapefile.Table(shapefile_path)

# Filter files for August only
august_files = filter(files) do file
    ds = Dataset(file)
    raw_time = ds.attrib["time_coverage_start"]
    dt = DateTime(raw_time[1:10], dateformat"yyyy-mm-dd")
    close(ds)
    return Dates.month(dt) == 8
end

# Log-scale colorbar ticks/labels (in original units)
cb_ticks = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50]
cb_tick_pos = log10.(cb_ticks)
cb_tick_labels = string.(cb_ticks)

# Use coalesce to handle 'missing' values in the 'name' column
erie_geom = [row.geometry for row in lakes if coalesce(row.name, "") == "Lake Erie"]
anim = @animate for file in august_files
    ds = Dataset(file)
    
    # --- Data Preparation ---
    data = coalesce.(ds["chlor_a"][lon_indices, lat_indices], NaN)
    data_sorted = data[p_lon, p_lat]
    data_sorted[.!isfinite.(data_sorted)] .= 0.1
    z = log10.(data_sorted)

    # --- 1. Create the Heatmap (Base Layer) ---
    # We use 'heatmap' (no bang) to start a fresh frame
    heatmap(
        lons, lats, z';
        xlims = (-83.6, -78.2),
        ylims = (41.3, 43.1),
        xlabel = "Longitude",
        ylabel = "Latitude",
        title = "Lake Erie Chlorophyll: $(get_pretty_date(ds))",
        aspect_ratio = :equal,
        color = :algae,
        clims = (log10(0.1), log10(50.0)),
        colorbar_ticks = (cb_tick_pos, cb_tick_labels),
        colorbar_title = "Chlorophyll-a log(mg/m³)",
        xticks = (xtick_vals, xtick_labels),
        yticks = (ytick_vals, ytick_labels),
        legend = false
    )

    # --- 2. Overlay the Shapefile (Top Layer) ---
    # We use 'plot!' (with bang) to draw on the heatmap
    plot!(
        erie_geom, 
        linewidth = 1.2, 
        linecolor = :black, 
        fillalpha = 0, 
        label = "",
        colorbar=true
    )

    close(ds)
end

mp4(anim, "lake_erie_august.mp4", fps = 5)

[ Info: Saved animation to c:\Users\kavin\OneDrive\Research\CPS\cdc26_ive_intro\code\habs\lake_erie_august.mp4


Plots.AnimatedGif("c:\\Users\\kavin\\OneDrive\\Research\\CPS\\cdc26_ive_intro\\code\\habs\\lake_erie_august.mp4")